In [ ]:
import pandas as pd
import numpy as np

# Data Processing


In [ ]:
import kagglehub
kagglehub.login()
path = kagglehub.competition_download('datathon-2026-round-1')
print("Data downloaded to:", path)

UnauthenticatedError: User is not authenticated

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import os
import shutil

# Define source and destination
source_dir = path
dest_dir = '/content/'

# Move all files from the downloaded directory to /content
for filename in os.listdir(source_dir):
    file_path = os.path.join(source_dir, filename)
    if os.path.isfile(file_path):
        shutil.copy(file_path, dest_dir)
        print(f"Moved: {filename}")

NameError: name 'path' is not defined

In [ ]:
BASE_PATH = '/content/'

customers = pd.read_csv(BASE_PATH + 'customers.csv')
geography = pd.read_csv(BASE_PATH + 'geography.csv')
inventory = pd.read_csv(BASE_PATH + 'inventory.csv')
orders = pd.read_csv(BASE_PATH + 'orders.csv')
order_items = pd.read_csv(BASE_PATH + 'order_items.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')
promotions = pd.read_csv(BASE_PATH + 'promotions.csv')
returns = pd.read_csv(BASE_PATH + 'returns.csv')
reviews = pd.read_csv(BASE_PATH + 'reviews.csv')
sales = pd.read_csv(BASE_PATH + 'sales.csv')
sample_submission = pd.read_csv(BASE_PATH + 'sample_submission.csv')
shipments = pd.read_csv(BASE_PATH + 'shipments.csv')
web_traffic = pd.read_csv(BASE_PATH + 'web_traffic.csv')

/tmp/ipykernel_5135/1091819555.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(BASE_PATH + 'order_items.csv')


In [ ]:
import pandas as pd
import numpy as np

# Định nghĩa hàm load để tái sử dụng và kiểm soát lỗi
def load_and_prep_data():
    # Parse_dates giúp chuyển các cột thời gian từ string sang datetime ngay lập tức
    orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
    products = pd.read_csv('products.csv')
    payments = pd.read_csv('payments.csv')
    customers = pd.read_csv('customers.csv')
    order_items = pd.read_csv('order_items.csv')
    shipments = pd.read_csv('shipments.csv', parse_dates=['ship_date', 'delivery_date'])
    web_traffic = pd.read_csv('web_traffic.csv', parse_dates=['date'])
    returns = pd.read_csv('returns.csv')
    reviews = pd.read_csv('reviews.csv')
    geo = pd.read_csv('geography.csv')
    inventory = pd.read_csv('inventory.csv', parse_dates=['snapshot_date'])

    # 2. Xử lý các giá trị Null cơ bản
    # Thay thế age_group null bằng 'Unknown' để không bị mất dữ liệu khi group
    customers['age_group'] = customers['age_group'].fillna('Unknown')

    # 3. Tạo các cột thời gian phái sinh (Feature Engineering)
    orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
    orders['year'] = orders['order_date'].dt.year

    # 4. Thực hiện các bước JOIN chính theo Cardinality
    # Join 1:1:1 (Orders - Payments - Customers)
    df_master = orders.merge(payments, on='order_id', how='left')
    df_master = df_master.merge(customers, on='customer_id', how='left')

    # Join Logistics (Shipments - Reviews - Geo)
    # Lưu ý: Zip trong orders thường dùng để giao hàng
    df_logistics = shipments.merge(reviews, on='order_id', how='left')
    df_logistics = df_logistics.merge(orders[['order_id', 'zip']], on='order_id', how='left')
    df_logistics = df_logistics.merge(geo, on='zip', how='left')

    # Join Inventory & Products
    df_inventory_master = inventory.merge(products, on='product_id', how='left')

    return {
        'orders': orders,
        'products': products,
        'payments': payments,
        'customers': customers,
        'order_items': order_items,
        'shipments': shipments,
        'web_traffic': web_traffic,
        'returns': returns,
        'reviews': reviews,
        'geo': geo,
        'inventory': inventory,
        'df_master': df_master,
        'df_logistics': df_logistics,
        'df_inventory_master': df_inventory_master
    }

# Gọi hàm để lấy dữ liệu
data = load_and_prep_data()

# Giải nén các dataframe để dùng trực tiếp như các phần code trước
orders = data['orders']
products = data['products']
payments = data['payments']
customers = data['customers']
order_items = data['order_items']
df_master = data['df_master']
df_logistics = data['df_logistics']
web_traffic = data['web_traffic']
returns = data['returns']
# ... tiếp tục cho các bảng khác

/tmp/ipykernel_5135/1268186650.py:11: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('order_items.csv')


In [ ]:
# Display summary statistics for the orders dataframe
display(orders.describe(include='all'))

# Display basic info including non-null counts and data types
print('\nDataFrame Info:')
display(orders.info())

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,month,year
count,646945.000000,646945,646945.000000,646945.000000,646945,646945,646945,646945,646945,646945.000000
unique,NaN,NaN,NaN,NaN,6,5,3,6,126,NaN
top,NaN,NaN,NaN,NaN,delivered,credit_card,mobile,organic_search,2018-06,NaN
freq,NaN,NaN,NaN,NaN,516716,356352,291482,181495,10851,NaN
mean,417189.470332,2016-11-28 05:46:39.463323904,84906.203535,55410.740423,NaN,NaN,NaN,NaN,NaN,2016.410148
min,1.000000,2012-07-04 00:00:00,1.000000,1001.000000,NaN,NaN,NaN,NaN,NaN,2012.000000
25%,208728.000000,2014-08-05 00:00:00,41336.000000,30904.000000,NaN,NaN,NaN,NaN,NaN,2014.000000
50%,417211.000000,2016-07-25 00:00:00,87279.000000,54129.000000,NaN,NaN,NaN,NaN,NaN,2016.000000
75%,625628.000000,2018-08-27 00:00:00,133282.000000,83301.000000,NaN,NaN,NaN,NaN,NaN,2018.000000
max,834397.000000,2022-12-31 00:00:00,157563.000000,99950.000000,NaN,NaN,NaN,NaN,NaN,2022.000000



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
 8   month           646945 non-null  object        
 9   year            646945 non-null  int32         
dtypes: datetime64[ns](1), int32(1), int64(3), object(5)
memory usage: 46.9+ MB


None

#Question 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A) 30 ngày
------
B) 90 ngày
------
C) 144 ngày
------
D) 365 ngày
------

In [ ]:

# Q1: Trung vị inter-order gap
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_sorted = orders.sort_values(['customer_id', 'order_date'])
orders_sorted['gap'] = orders_sorted.groupby('customer_id')['order_date'].diff().dt.days
q1_val = orders_sorted['gap'].dropna().median()
print(f"Q1 - Trung vị Inter-order gap: {q1_val} ngày")

Q1 - Trung vị Inter-order gap: 144.0 ngày


# -> Đáp án C: xấp xỉ 144 ngày



# Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

A) Premium
------
B) Performance
------
C) Activewear
------
D) Standard
------

In [ ]:
# Q2. Segment có tỷ suất lợi nhuận gộp (gross margin) trung bình cao nhất
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
q2_val = products.groupby('segment')['gross_margin'].mean().idxmax()
print(f"Q2 - Segment lợi nhuận cao nhất: {q2_val}")

Q2 - Segment lợi nhuận cao nhất: Standard


# Đáp án : D. Standard

# Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective
-----
B) wrong_size
-----
C) changed_mind
-----
D) not_as_described
-----

In [ ]:
# Q3. Lý do trả hàng (return_reason) phổ biến nhất của Streetwear
st_returns = returns.merge(products[['product_id', 'category']], on='product_id')
q3_val = st_returns[st_returns['category'] == 'Streetwear']['return_reason'].mode()[0]
print(f"Q3 - Lý do trả hàng Streetwear phổ biến nhất: {q3_val}")

Q3 - Lý do trả hàng Streetwear phổ biến nhất: wrong_size


# Đáp án: B. wrong_size

# Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

A) organic_search
------
B) paid_search
------
C) email_campaign
------
D) social_media
------

In [ ]:
# Q4. Nguồn truy cập (traffic_source) có bounce_rate thấp nhất
q4_val = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4 - Nguồn traffic có Bounce Rate thấp nhất: {q4_val}")

Q4 - Nguồn traffic có Bounce Rate thấp nhất: email_campaign


# Đáp án: C. email_campaign

# Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

A) 12%
-----
B) 25%
-----
C) 39%
-----
D) 54%
-----

In [ ]:
# Q5. Tỷ lệ % dòng trong order_items có áp dụng khuyến mãi
q5_val = (order_items['promo_id'].notnull().sum() / len(order_items)) * 100
print(f"Q5 - Tỷ lệ đơn hàng có khuyến mãi: {q5_val:.2f}%")

Q5 - Tỷ lệ đơn hàng có khuyến mãi: 38.66%


# ~ 39%
# Đáp án: C. 39%

# Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

A) 55+
-----
B) 25–34
-----
C) 35–44
-----
D) 45–54
-----

In [ ]:
# Q6. Nhóm tuổi có số đơn hàng trung bình/khách hàng cao nhất
cust_orders = orders.merge(customers[['customer_id', 'age_group']], on='customer_id')
age_stats = cust_orders.groupby('age_group').agg(
    total_orders=('order_id', 'nunique'),
    total_cust=('customer_id', 'nunique')
)
q6_val = (age_stats['total_orders'] / age_stats['total_cust']).idxmax()
print(f"Q6 - Nhóm tuổi có số đơn hàng TB cao nhất: {q6_val}")

Q6 - Nhóm tuổi có số đơn hàng TB cao nhất: 55+


# Đáp án: A. 55+

# Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

A) West
-----
B) Central
-----
C) East
-----
D) Cả ba vùng có doanh thu xấp xỉ bằng nhau
-----

In [ ]:
# Q7. Vùng (region) tạo ra tổng doanh thu cao nhất
pay_orders = payments.merge(orders[['order_id', 'zip']], on='order_id')
region_revenue = pay_orders.merge(geography[['zip', 'region']], on='zip')
q7_val = region_revenue.groupby('region')['payment_value'].sum().idxmax()
print(f"Q7 - Vùng có doanh thu cao nhất: {q7_val}")

Q7 - Vùng có doanh thu cao nhất: East


# Đáp án: *C. East*

# Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

A) credit_card
------
B) cod
------
C) paypal
------
D) bank_transfer
------

In [ ]:
# Q8. Phương thức thanh toán của đơn hàng 'cancelled' phổ biến nhất
cancelled_orders = orders[orders['order_status'] == 'cancelled']
q8_val = cancelled_orders['payment_method'].mode()[0]
print(f"Q8 - Phương thức thanh toán phổ biến cho đơn huỷ: {q8_val}")

Q8 - Phương thức thanh toán phổ biến cho đơn huỷ: credit_card


# Đáp án: *A. credit_card*

# Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

A) S
-----
B) M
-----
C) L
-----
D) XL
-----

In [ ]:
# Q9. Kích thước (size) có tỷ lệ trả hàng cao nhất
sales_by_size = order_items.merge(products[['product_id', 'size']], on='product_id')['size'].value_counts()
returns_by_size = returns.merge(products[['product_id', 'size']], on='product_id')['size'].value_counts()
q9_val = (returns_by_size / sales_by_size).idxmax()
print(f"Q9 - Kích thước có tỷ lệ trả hàng cao nhất: {q9_val}")

Q9 - Kích thước có tỷ lệ trả hàng cao nhất: S


# Đáp án: *A. S*

# Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

A) 1 kỳ (trả một lần)
------
B) 3 kỳ
------
C) 6 kỳ
------
D) 12 kỳ
------

In [ ]:
# Q10. Kế hoạch trả góp (installments) có giá trị thanh toán trung bình cao nhất
q10_val = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f"Q10 - Số kỳ trả góp có giá trị trung bình cao nhất: {q10_val}")

Q10 - Số kỳ trả góp có giá trị trung bình cao nhất: 6


# Đáp án: *C. 6 kỳ*